# Análise dos resultados — MLP Dry Bean

Este notebook lê os CSVs gerados por `01_experimentos.ipynb` e produz tabelas Markdown em `results/tables/` para o relatório. Ele não treina modelos.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RESULTS_CSV = PROJECT_ROOT / "results" / "csv"
RESULTS_TAB = PROJECT_ROOT / "results" / "tables"
RESULTS_TAB.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

def df_to_markdown(df, floatfmt=".4f"):
    """Converte DataFrame em tabela Markdown sem dependências extras."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |",
             "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        vals = []
        for v in row:
            if isinstance(v, (float, np.floating)):
                if np.isnan(v):
                    vals.append("—")
                else:
                    vals.append(format(float(v), floatfmt))
            else:
                vals.append(str(v).replace("|", "\\|"))
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)

def save_table(df, name, floatfmt=".4f"):
    path = RESULTS_TAB / name
    path.write_text(df_to_markdown(df, floatfmt) + "\n", encoding="utf-8")
    print("wrote", path)

def load(name, required=True):
    p = RESULTS_CSV / name
    if not p.exists():
        if required:
            print("AUSENTE:", p, "— execute 01_experimentos.ipynb")
            return None
        return None
    return pd.read_csv(p)

print("Resultados:", RESULTS_CSV)

## Q1 — Inicializações

In [ ]:
q1 = load("q1_runs.csv")
if q1 is not None:
    cols = ["run_id", "seed", "epochs", "loss_final", "accuracy_final",
            "loss_min", "loss_min_epoch", "macro_f1_final", "time_s"]
    t = q1[cols].sort_values("seed")
    save_table(t, "q1_tabela.md")
    display(t.round(4))
    print("Média loss final:", q1["loss_final"].mean())
    print("Desvio-padrão loss final:", q1["loss_final"].std())
    print("Média acurácia final:", q1["accuracy_final"].mean())
    print("Desvio-padrão acurácia final:", q1["accuracy_final"].std())

## Q2 — Learning rate e momento

In [ ]:
q2 = load("q2_grid.csv")
if q2 is not None:
    cols = ["run_id", "learning_rate", "momentum", "status", "converged", "epochs",
            "loss_train", "acc_train", "loss_val", "acc_val", "macro_f1_val",
            "weighted_f1_val", "time_s", "selected"]
    t = q2[cols].sort_values(["selected", "macro_f1_val"], ascending=[False, False])
    save_table(t, "q2_tabela.md")
    display(t.round(4))

## Q3 — Topologia

In [ ]:
q3 = load("q3_grid.csv")
if q3 is not None:
    cols = ["rank", "run_id", "n_layers", "neurons", "n_params", "status", "converged",
            "epochs", "loss_train", "acc_train", "loss_val", "acc_val",
            "macro_f1_val", "weighted_f1_val", "gap", "time_s", "selected_top4"]
    t = q3[cols].sort_values(["rank", "n_layers", "neurons"], na_position="last")
    save_table(t, "q3_tabela.md")
    display(t.round(4))
    top4 = q3[q3["selected_top4"] == True].sort_values("rank")
    save_table(top4[cols], "q3_top4_tabela.md")

## Q4 — Quantidade de dados

In [ ]:
q4 = load("q4_fractions.csv")
if q4 is not None:
    cols = ["run_id", "fraction", "n_train", "status", "converged", "epochs",
            "loss_train", "acc_train", "loss_val", "acc_val", "macro_f1_val",
            "weighted_f1_val", "time_s", "selected"]
    t = q4[cols].sort_values("fraction")
    save_table(t, "q4_tabela.md")
    display(t.round(4))

## Q5 — Atributos

In [ ]:
q5f = load("q5_features.csv")
if q5f is not None:
    cols = ["feature", "variance", "mutual_info", "all_16", "low_redundancy",
            "top_10_mi", "compact", "no_noise"]
    t = q5f[cols].sort_values("mutual_info", ascending=False)
    save_table(t, "q5_features_tabela.md")
    display(t.round(5))

q5 = load("q5_subsets.csv")
if q5 is not None:
    cols = ["run_id", "subset", "n_features", "status", "reused", "converged",
            "epochs", "loss_train", "acc_train", "loss_val", "acc_val",
            "macro_f1_val", "weighted_f1_val", "time_s", "selected"]
    t = q5[cols].sort_values(["selected", "macro_f1_val"], ascending=[False, False])
    save_table(t, "q5_subsets_tabela.md")
    display(t.round(4))

## Q6 — Validação final no teste

In [ ]:
q6 = load("q6_test.csv")
if q6 is not None:
    cols = ["run_id", "n_layers", "neurons", "n_params", "status", "converged",
            "epochs", "loss_train", "acc_train", "macro_f1_train", "loss_test",
            "acc_test", "macro_f1_test", "weighted_f1_test", "precision_macro_test",
            "recall_macro_test", "time_s", "selected"]
    t = q6[cols].sort_values(["selected", "macro_f1_test"], ascending=[False, False])
    save_table(t, "q6_tabela.md")
    display(t.round(4))

q6r = load("q6_classification_report.csv", required=False)
if q6r is not None:
    q6r_md = q6r.reset_index().rename(columns={"index": "class"})
    save_table(q6r_md, "q6_classification_report.md", floatfmt=".4f")
    display(q6r_md.round(4))

## Q7 — Validação cruzada

In [ ]:
q7 = load("q7_cv.csv")
if q7 is not None:
    cols = ["fold", "status", "converged", "epochs", "n_train", "n_val",
            "loss_train", "acc_train", "loss_val", "acc_val", "macro_f1_val",
            "weighted_f1_val", "precision_macro_val", "recall_macro_val", "time_s"]
    t = q7[cols].sort_values("fold")
    save_table(t, "q7_folds_tabela.md")
    display(t.round(4))

q7s = load("q7_summary.csv")
if q7s is not None:
    save_table(q7s, "q7_summary_tabela.md", floatfmt=".6f")
    display(q7s.round(6))

## Consolidado dos vencedores

In [ ]:
w = load("winners.csv")
if w is not None:
    save_table(w, "winners_tabela.md", floatfmt=".5f")
    display(w.round(5))
print("Tabelas geradas em", RESULTS_TAB)

## Encerramento

As tabelas acima devem ser copiadas para `report/relatorio.md`, sempre acompanhadas da análise correspondente. Nenhum valor deve ser alterado manualmente; a fonte é `results/csv/`.